<h3 style="color:#6FA8DC; font-weight:bold">03_Missing_Data_Arbitrary_Value_Imputation</h3>

Handling Missing Data → Univariate Numerical Imputation

Reference basis: the provided CampusX numerical-imputation notebooks and `titanic_toy.csv`.

<h5 style="color:#78B89A; font-weight:bold;">1. What is univariate imputation? → one feature at a time</h5>

Univariate imputation fills missing values in a feature using information from **that same feature**.

Example:

`Age = [22, 38, NaN, 35, 28]`

The missing `Age` is filled using a rule based on the observed `Age` values.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

In [ ]:
df = pd.read_csv('titanic_toy.csv')
df.head()

In [ ]:
df.isnull().mean() * 100

In [ ]:
X = df.drop(columns=['Survived'])
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2
)

X_train.isnull().mean() * 100

<h5 style="color:#78B89A; font-weight:bold;">2. Why impute after train/test split? → avoid leakage</h5>

The value used to fill missing data must be **learned from the training set only**.

```text
X_train → learn imputation value
X_train → transform
X_test  → use the SAME learned value
```

Never calculate a mean/median using the complete dataset before splitting. That allows information from the test set to influence training.

<h5 style="color:#78B89A; font-weight:bold;">3. Method: Arbitrary Value Imputation</h5>

<h5 style="color:#78B89A; font-weight:bold;">4. Idea → replace NaN with a fixed value</h5>

Instead of calculating a statistic, choose a constant that represents missingness.

The reference notebook demonstrates values such as **99 / 999** and **-1** for the Titanic `Age` and `Fare` variables.

In [ ]:
X_train['Age_99'] = X_train['Age'].fillna(99)
X_train['Age_minus1'] = X_train['Age'].fillna(-1)
X_train['Fare_999'] = X_train['Fare'].fillna(999)
X_train['Fare_minus1'] = X_train['Fare'].fillna(-1)

X_train[['Age','Age_99','Age_minus1','Fare','Fare_999','Fare_minus1']].head()

<h5 style="color:#78B89A; font-weight:bold;">5. Why use an arbitrary value? → create an explicit missing code</h5>

A value outside the normal range can make missingness distinguishable from ordinary observations.

For example, if `Age` normally ranges from 0 to 80, `-1` is clearly not a real age.

<h5 style="color:#78B89A; font-weight:bold;">6. Important warning → choose the constant carefully</h5>

The chosen value should make sense for the feature and model.

Do not blindly use 999 for every variable. A bad constant can:

- create extreme outliers,
- distort the distribution,
- affect distance-based models,
- affect linear models,
- make the feature scale very unusual.

In [ ]:
print('Original Age variance:', X_train['Age'].var())
print('Age variance after -1:', X_train['Age_minus1'].var())
print('Age variance after 99:', X_train['Age_99'].var())

fig, ax = plt.subplots(figsize=(8,4))
X_train['Age'].plot(kind='kde', ax=ax, label='Original')
X_train['Age_minus1'].plot(kind='kde', ax=ax, label='-1')
X_train['Age_99'].plot(kind='kde', ax=ax, label='99')
ax.legend(); ax.set_title('Age: Arbitrary Value Imputation'); plt.show()

<h5 style="color:#78B89A; font-weight:bold;">7. When to use? → practical cases</h5>

- When missingness itself may carry information.
- When a safe out-of-range value exists.
- When you want a simple, deterministic rule.
- Often paired with a missing indicator when you want the model to explicitly know that the value was originally missing.

<h5 style="color:#78B89A; font-weight:bold;">8. Advantages / disadvantages</h5>

**Advantages:** simple, deterministic, preserves all rows, can encode missingness distinctly.

**Disadvantages:** can distort distributions and variance; a poor constant can behave like an outlier; the constant is domain-dependent.

<h5 style="color:#78B89A; font-weight:bold;">9. Modern scikit-learn way → SimpleImputer(strategy='constant')</h5>

In [ ]:
imputer = SimpleImputer(strategy='constant', fill_value=-1)

X_train_imp = imputer.fit_transform(X_train[['Age', 'Fare']])
X_test_imp = imputer.transform(X_test[['Age', 'Fare']])

print('Learned statistics:', imputer.statistics_)

<h5 style="color:#78B89A; font-weight:bold;">10. Modern production pattern → Pipeline</h5>

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value=-1)),
    ('model', LogisticRegression(max_iter=1000))
])

pipe.fit(X_train[['Age','Fare']], y_train)
# predictions = pipe.predict(X_test[['Age','Fare']])

<h5 style="color:#78B89A; font-weight:bold;">11. Final revision</h5>

```text
NaN
 ↓
Choose safe domain-specific constant
 ↓
SimpleImputer(strategy='constant')
 ↓
Fit on train
 ↓
Transform test + production
```

**Remember:** arbitrary imputation is powerful only when the chosen value has a meaningful reason.